Dataset Link: https://www.kaggle.com/datasets/mirzayasirabdullah07/student-exam-scores-dataset

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

In [3]:
df = pd.read_csv("student_exam_scores.csv")

df.drop(['student_id'], inplace=True, axis=1)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   hours_studied       200 non-null    float64
 1   sleep_hours         200 non-null    float64
 2   attendance_percent  200 non-null    float64
 3   previous_scores     200 non-null    int64  
 4   exam_score          200 non-null    float64
dtypes: float64(4), int64(1)
memory usage: 7.9 KB


In [5]:
df.describe()

,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,6.325500,6.622000,74.830000,66.800000,33.955000
std,3.227317,1.497138,14.249905,15.663869,6.789548
min,1.000000,4.000000,50.300000,40.000000,17.100000
25%,3.500000,5.300000,62.200000,54.000000,29.500000
50%,6.150000,6.700000,75.250000,67.500000,34.050000
75%,9.000000,8.025000,87.425000,80.000000,38.750000
max,12.000000,9.000000,100.000000,95.000000,51.300000


## Part A : Correlation Test

In [6]:
for col in (x for x in df.columns if x != "exam_score"):
    res = stats.pearsonr(x=df[col], y=df["exam_score"])
    print(f"For columns {col} and exam_score")
    print(f"\tr value is {res.statistic}")
    print(f"\tp value is {res.pvalue}")

For columns hours_studied and exam_score
	r value is 0.7767514349789608
	p value is 1.2719809567612592e-41
For columns sleep_hours and exam_score
	r value is 0.18822198470447815
	p value is 0.007606049140258496
For columns attendance_percent and exam_score
	r value is 0.22571260459020465
	p value is 0.0013107138203185325
For columns previous_scores and exam_score
	r value is 0.4311047112479791
	p value is 1.8582419062525198e-10


from above we conclude that 

hours studies is highly related to the exam score

attendence_percent, sleep_hours, and previous_score is not related tp exam score

## Part B : Z-Test

In [7]:
group1 = df[df["attendance_percent"] >= 75]["exam_score"]
group2 = df[df["attendance_percent"] < 75]["exam_score"]
df["group1"] = group1
df["group2"] = group2

df

,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score,group1,group2
0,8.0,8.8,72.1,45,30.2,NaN,30.2
1,1.3,8.6,60.7,55,25.0,NaN,25.0
2,4.0,8.2,73.7,86,35.8,NaN,35.8
3,3.5,4.8,95.1,66,34.0,34.0,NaN
4,9.1,6.4,89.8,71,40.3,40.3,NaN
...,...,...,...,...,...,...,...
195,10.5,5.4,94.0,87,42.7,42.7,NaN
196,7.1,6.1,85.1,92,40.4,40.4,NaN
197,1.6,6.9,63.8,76,28.2,NaN,28.2
198,12.0,7.3,50.5,58,42.0,NaN,42.0


In [8]:
print("Assumption: There is no Difference between the exam scores of students with high attendence (>= 75%) and low attendence (< 75%)")

print(f"Null Hypothesis: There is no Difference Between Groups")
print(f"Alternate Hypothesis: There is a difference between the scores")

t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=False)

print("T-statistic:", t_stat)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print("Reject the null hypothesis: Significant difference between groups.")
else:
    print("Fail to reject the null hypothesis: No significant difference.")

Assumption: There is no Difference between the exam scores of students with high attendence (>= 75%) and low attendence (< 75%)
Null Hypothesis: There is no Difference Between Groups
Alternate Hypothesis: There is a difference between the scores
T-statistic: 2.1390202343996547
P-value: 0.033668107851722036
Reject the null hypothesis: Significant difference between groups.


## Part C : Chi-Square Test

In [9]:
# Binning to get categorical dta
df["Grade"] = pd.cut(
    df["exam_score"],
    bins=[0, 25, 35, 45, 100],
    labels=["D", "C", "B", "A"]
)

observed = df["Grade"].value_counts().sort_index()
print(observed)

# Computing frequencies
expected = [observed.sum() / len(observed)] * len(observed)
print(expected)

Grade
D    21
C    91
B    76
A    12
Name: count, dtype: int64
[np.float64(50.0), np.float64(50.0), np.float64(50.0), np.float64(50.0)]


In [ ]:
print("Null Hypothesis: Exam Scores are equally distibuted accross the categories")
print("Alternate Hypothesis: Exam Scores ate not equally distributed\n")

chi2_stat, p_value = stats.chisquare(observed, expected)

print("Chi-square statistic:", chi2_stat)
print("P-value:", p_value, "\n")

alpha = 0.05

if p_value < alpha:
    print("Result is statistically significant ==> Reject Null Hypothesis")
else:
    print("Result is not statistically significant ==> Accept Null Hypothesis")

Null Hypothesis: Exam Scores are equally distibuted accross the categories
Alternate Hypothesis: Exam Scores ate not equally distributed

Chi-square statistic: 92.83999999999999
P-value: 5.3760379100944477e-20 

Result is statistically significant → Reject Null Hypothesis


In [11]:
import numpy as np

# Create a proper group label
df["Attendance_Group"] = np.where(
    df["attendance_percent"] >= 75,
    "High Attendance",
    "Low Attendance"
)

# Keep only clean columns
df_clean = df[
    ["hours_studied",
     "sleep_hours",
     "attendance_percent",
     "previous_scores",
     "exam_score",
     "Attendance_Group",
     "Grade"]
]

# Export clean file
df_clean.to_csv("student_exam_FINAL.csv", index=False)